# 78. 사람의 화면상 크기와 원판 탐색 여유는 선택을 바꾸는가?

기존 검증 6명의 원본을 첫 프레임부터 확인한다. 대조군은 사람 박스 면적 1.25배·원판 여유 15%다. 사람 면적만 1.10·1.50배로 바꾸거나 여유만 0·10%로 바꾼다. 각 조건에서 직전 수행자와 원판 이력을 새로 시작한다.

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
from wlcv.config import ROOT
from wlcv.evaluation.foreground_thresholds import collect, compare, plot_change

folder = ROOT / 'outputs' / 'foreground_thresholds'

### 같은 YOLO 후보 준비

기존 39장 검증 라벨이 있는 WL_DATASET_DIR, 원본 MP4가 있는 WL_VIDEO_DIR, 배포 가중치가 있는 WL_MODEL_DIR를 사용한다. 아래 셀은 CPU에서 영상당 한 번만 추론해 후보 좌표를 로컬에 저장한다. 같은 입력과 가중치로 이미 저장했다면 다음 셀부터 실행할 수 있다.

In [ ]:
collect(folder / 'candidates')

### 동작 중 수행자와 원판 채점

수행자 판독 박스와 IoU 0.50 이상 겹치는 후보가 둘 이상이면 복수 일치로 센다. 원판 정답이 있는 프레임만 미탐·오탐을 센다. 순서대로 동작 전(before), 동작 중(motion), 동작 후(after)를 확인할 수 있다.

In [ ]:
summary, frames = compare(folder)
display(summary[summary.phase == 'motion'])
display(summary[summary.phase != 'motion'])

### 선택이 달라진 장면

초록색은 선택한 사람, 점선은 다른 사람 후보, 주황색은 원판이다. output_changed는 수행자나 최종 원판 결과가 달라진 장면, pool_changed는 추적기에 들어가기 전 원판 후보 목록이 달라진 장면이다.

In [ ]:
changed = frames[frames.output_changed]
columns = ['condition', 'video_id', 'frame_index', 'phase', 'selected_control', 'selected', 'reason', 'person_review', 'plate_fn', 'plate_fp']
display(changed[columns])
display(frames[frames.pool_changed][['condition', 'video_id', 'frame_index', 'plate_changed']])
for row in changed.itertuples():
    plot_change(folder, frames, row.video_id, row.frame_index, row.condition)
    plt.show()